# Vito / NRP 046 — YOLO11s-seg — small

Train the **already annotated Panoptic Indoor Segmentation v5** now. Annotate your own photos later and use them only for held-out testing.

**Before Run All:** enable Kaggle GPU + Internet. Add the secret `ROBOFLOW_API_KEY` and enable access, or set `DATA_ROOT` to an attached **v5 COCO Segmentation** export. No secret is embedded in this notebook.

This notebook runs **3 separate augmentation conditions × 30 epochs** by default. Across the three model notebooks, that is **9 fits**, covering the digit-6 requirement. To stage the work, use `REGIMES = ["default"]` first, then run the other conditions later. An augmentation condition always starts from pretrained initialization unless resuming that exact condition.

Packages are pinned below; keep Kaggle's CUDA torch/torchvision pair. Save notebook outputs between sessions. CPU synthetic checks passed during preparation; actual v5 GPU training has not been run here.


## 1. Install dependencies

In [ ]:
import os, sys, subprocess
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"
os.environ["WANDB_MODE"] = "disabled"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "ultralytics==8.3.221", "albumentations==2.0.8", "pycocotools==2.0.10",
    "roboflow==1.2.11", "PyYAML==6.0.2"])
# Keep Kaggle's matched CUDA torch/torchvision versions; do not install CPU wheels.


## 2. Configuration

In [ ]:
# EDIT ONLY THESE SETTINGS BEFORE STARTING.
MODEL = 'yolo11s-seg'
EPOCHS = 30                    # 25 is also fine: use the same value for all compared runs.
REGIMES = ["default", "none", "custom"]  # 3 independent fits in this notebook.
IMGSZ = 640
YOLO_BATCH = 4
MASK_BATCH = 2
WORKERS = 2                   # Use 0 if Kaggle reports a multiprocessing error.
DATA_ROOT = ""                # Empty = automatic exact-v5 download using Kaggle secret.
STRICT_V5 = True
OUT_ROOT = "/kaggle/working/seg046/runs"
PREPARED_ROOT = "/kaggle/working/seg046/prepared"
RESTORE_RUNS_FROM = ""         # Optional /kaggle/input/... folder containing prior run folders.
# Restore only your own trusted checkpoints. For an uploaded ZIP, extract it into a
# working directory first, then point RESTORE_RUNS_FROM at its parent run directory.


## 3. Self-contained implementation
This cell embeds the shared code; no separate script upload is needed.

In [ ]:
from pathlib import Path
import sys, importlib
CODE_DIR = Path("/kaggle/working/seg046_code")
CODE_DIR.mkdir(parents=True, exist_ok=True)
(CODE_DIR / "seg046.py").write_text('"""NRP 046: Kaggle instance-segmentation training and held-out evaluation.\nSource-of-truth labels are COCO instance masks; all model outputs use COCOeval.\n"""\nfrom __future__ import annotations\nimport contextlib, csv, hashlib, io, json, os, random, shutil, subprocess, sys, time\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\nimport numpy as np\nfrom PIL import Image, ImageDraw\n\nSEED = 46\nMODELS = (\'yolo11n-seg\', \'yolo11s-seg\', \'maskrcnn\')\nREGIMES = (\'none\', \'default\', \'custom\')\nDATASET_URL = \'https://universe.roboflow.com/panopticindoor/panoptic-indoor-segmentation/dataset/5\'\nEXPECTED_COUNTS = {\'train\': 397, \'valid\': 166, \'test\': 359}\nYOLO_AUG = {\n    \'none\': dict(hsv_h=0., hsv_s=0., hsv_v=0., degrees=0., translate=0., scale=0.,\n                 shear=0., perspective=0., flipud=0., fliplr=0., mosaic=0., mixup=0.,\n                 copy_paste=0., cutmix=0., bgr=0., close_mosaic=0),\n    \'default\': dict(hsv_h=.015, hsv_s=.7, hsv_v=.4, degrees=0., translate=.1, scale=.5,\n                    shear=0., perspective=0., flipud=0., fliplr=.5, mosaic=1., mixup=0.,\n                    copy_paste=0., cutmix=0., bgr=0., close_mosaic=10),\n    \'custom\': dict(hsv_h=.01, hsv_s=.3, hsv_v=.6, degrees=10., translate=.08, scale=.2,\n                   shear=3., perspective=0.0002, flipud=0., fliplr=.5, mosaic=.25,\n                   mixup=0., copy_paste=.1, cutmix=0., bgr=0., close_mosaic=10),\n}\n\ndef write_json(path, data):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + \'.tmp\')\n    tmp.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding=\'utf-8\')\n    tmp.replace(path)\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for b in iter(lambda: f.read(1024 * 1024), b\'\'): h.update(b)\n    return h.hexdigest()\n\ndef seed_all(seed=SEED):\n    import torch\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n\ndef seed_worker(worker_id):\n    import torch\n    s = torch.initial_seed() % 2**32\n    random.seed(s); np.random.seed(s)\n\ndef require_gpu():\n    import torch, torchvision\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'Enable a GPU accelerator in Kaggle Settings before training.\')\n    # Also fail early if Kaggle torch/torchvision are incompatible.\n    torchvision.ops.nms(torch.tensor([[0.,0.,1.,1.]],device=\'cuda\'),\n                        torch.tensor([1.],device=\'cuda\'), .5)\n    print(\'GPU:\', torch.cuda.get_device_name(0), \'| torch:\', torch.__version__,\n          \'| torchvision:\', torchvision.__version__)\n\ndef obtain_dataset(data_root=\'\', download_root=\'/kaggle/working/panoptic_v5\'):\n    if data_root:\n        root = Path(data_root)\n    else:\n        root = Path(download_root)\n        if not list(root.rglob(\'_annotations.coco.json\')):\n            from kaggle_secrets import UserSecretsClient\n            from roboflow import Roboflow\n            try:\n                key = UserSecretsClient().get_secret(\'ROBOFLOW_API_KEY\')\n            except Exception as exc:\n                raise RuntimeError(\'Add the Kaggle secret ROBOFLOW_API_KEY and enable it for this notebook, \'\n                                   \'or set DATA_ROOT to an attached COCO Segmentation v5 export.\') from exc\n            # Suppress SDK output: download URLs can contain the API key.\n            try:\n                with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):\n                    ds = (Roboflow(api_key=key).workspace(\'panopticindoor\')\n                          .project(\'panoptic-indoor-segmentation\').version(5)\n                          .download(\'coco-segmentation\', location=str(root), overwrite=False))\n                root = Path(ds.location)\n            except Exception:\n                raise RuntimeError(\'Roboflow download failed. Check Internet and the API-key permission, \'\n                                   \'or attach a manual v5 COCO Segmentation export as DATA_ROOT.\') from None\n    if not root.exists(): raise FileNotFoundError(root)\n    # Accommodate one extra directory level in a Kaggle dataset upload.\n    candidates = [p.parent.parent for p in root.rglob(\'_annotations.coco.json\') if p.parent.name==\'train\']\n    if (root/\'train/_annotations.coco.json\').exists(): return root\n    candidates = sorted(set(candidates))\n    if len(candidates) != 1:\n        raise ValueError(f\'Expected one COCO training export under {root}; found {len(candidates)}. Set DATA_ROOT precisely.\')\n    return candidates[0]\n\ndef decode_segmentation(seg, height, width):\n    from pycocotools import mask as mu\n    if isinstance(seg, list):\n        if not seg: raise ValueError(\'Missing instance polygon; a bounding-box export cannot train segmentation.\')\n        if any(len(p)<6 or len(p)%2 for p in seg): raise ValueError(\'Invalid COCO polygon.\')\n        rle = mu.merge(mu.frPyObjects(seg, height, width))\n    elif isinstance(seg, dict):\n        rle = mu.frPyObjects(seg, height, width) if isinstance(seg.get(\'counts\'), list) else dict(seg)\n        if isinstance(rle.get(\'counts\'), str): rle[\'counts\'] = rle[\'counts\'].encode()\n    else:\n        raise ValueError(\'Expected COCO instance polygons/RLE, not a semantic/panoptic PNG export.\')\n    m = mu.decode(rle)\n    if m.ndim == 3: m = np.any(m, axis=2)\n    if m.shape != (height, width): raise ValueError(\'Mask/image dimensions differ.\')\n    return m.astype(np.uint8)\n\ndef encode_mask(mask):\n    from pycocotools import mask as mu\n    rle = mu.encode(np.asfortranarray(mask.astype(np.uint8)))\n    rle[\'counts\'] = rle[\'counts\'].decode(\'ascii\')\n    return rle\n\ndef category_names(raw):\n    used = {a[\'category_id\'] for a in raw[\'annotations\']}\n    parents = {c.get(\'supercategory\') for c in raw[\'categories\']}\n    # Roboflow often adds an unused parent category, which is not a trainable class.\n    cats = [c for c in raw[\'categories\'] if c[\'id\'] in used or c[\'name\'] not in parents]\n    names = [c[\'name\'] for c in cats]\n    if len(names)!=len(set(names)): raise ValueError(\'Duplicate class names in COCO categories.\')\n    return sorted(names)\n\ndef find_image(json_path, record):\n    base = Path(json_path).parent\n    p = base / record[\'file_name\']\n    if not p.is_file(): p = base / Path(record[\'file_name\']).name\n    if not p.is_file(): raise FileNotFoundError(f\'Missing image {record["file_name"]} near {json_path}\')\n    return p.resolve()\n\ndef canonicalize(json_path, names):\n    from pycocotools import mask as mu\n    raw = read_json(json_path)\n    if not raw.get(\'images\'): raise ValueError(f\'Empty image split: {json_path}\')\n    if not raw.get(\'annotations\'): raise ValueError(f\'No annotations: {json_path}\')\n    ids = [x[\'id\'] for x in raw[\'images\']]\n    if len(ids)!=len(set(ids)): raise ValueError(\'Duplicate image IDs.\')\n    cat = {c[\'id\']:c[\'name\'] for c in raw[\'categories\']}\n    name_to_id = {n:i+1 for i,n in enumerate(names)}\n    images, paths, hashes = [], {}, {}\n    for rec in raw[\'images\']:\n        p = find_image(json_path, rec)\n        with Image.open(p) as im: w,h = im.size\n        if (w,h)!=(rec[\'width\'],rec[\'height\']):\n            raise ValueError(f\'Image size differs from COCO metadata: {p.name}. Re-export without changing the images.\')\n        images.append(dict(id=rec[\'id\'],file_name=str(p),width=w,height=h))\n        paths[rec[\'id\']] = (w,h)\n        hashes[rec[\'id\']] = sha256(p)\n    anns=[]\n    for ann in raw[\'annotations\']:\n        if ann[\'image_id\'] not in paths: raise ValueError(\'Annotation references an absent image.\')\n        name=cat[ann[\'category_id\']]\n        if name not in name_to_id:\n            raise ValueError(f\'Unknown class {name!r}. Match the training class spelling exactly; do not silently merge labels.\')\n        w,h=paths[ann[\'image_id\']]\n        m=decode_segmentation(ann.get(\'segmentation\'),h,w)\n        rle=mu.encode(np.asfortranarray(m))\n        area=float(mu.area(rle)); box=mu.toBbox(rle).tolist()\n        if area<=0: raise ValueError(f\'Empty instance mask, annotation {ann["id"]}. Repair the source annotation.\')\n        anns.append(dict(id=len(anns)+1,image_id=ann[\'image_id\'],category_id=name_to_id[name],\n                         segmentation=ann[\'segmentation\'],bbox=box,area=area,iscrowd=int(ann.get(\'iscrowd\',0))))\n    return dict(info={\'description\':\'Canonical instance masks; IDs remapped by exact class name\'},\n                licenses=raw.get(\'licenses\',[]),images=images,annotations=anns,\n                categories=[dict(id=i+1,name=n) for i,n in enumerate(names)]), hashes\n\ndef prepare_data(source_root, prepared_root, strict_v5=True):\n    root=Path(source_root); out=Path(prepared_root); out.mkdir(parents=True,exist_ok=True)\n    sources={s:root/s/\'_annotations.coco.json\' for s in (\'train\',\'valid\',\'test\')}\n    if not sources[\'valid\'].exists(): sources[\'valid\']=root/\'val/_annotations.coco.json\'\n    for p in sources.values():\n        if not p.exists(): raise FileNotFoundError(f\'Missing original split: {p}; do not substitute validation for test.\')\n    names=category_names(read_json(sources[\'train\']))\n    prepared={}; split_hashes={}; counts={}; class_counts={}; duplicate_pairs=[]\n    for s,p in sources.items():\n        canonical, hashes=canonicalize(p,names)\n        counts[s]=len(canonical[\'images\']); split_hashes[s]=set(hashes.values())\n        prepared[s]=str(out/f\'{s}.json\'); write_json(prepared[s],canonical)\n        class_counts[s]=dict(Counter(names[a[\'category_id\']-1] for a in canonical[\'annotations\']))\n    if strict_v5 and counts != EXPECTED_COUNTS:\n        raise ValueError(f\'Expected v5 split sizes {EXPECTED_COUNTS}, found {counts}. Check version/export; \'\n                         \'only set STRICT_V5=False for an explicitly documented exception.\')\n    for a,b in [(\'train\',\'valid\'),(\'train\',\'test\'),(\'valid\',\'test\')]:\n        overlap=split_hashes[a]&split_hashes[b]\n        if overlap: duplicate_pairs.append({\'splits\':[a,b],\'exact_duplicate_count\':len(overlap)})\n    # Preserve the assignment splits but expose any source duplication, never resplit silently.\n    if duplicate_pairs: print(\'WARNING: source split duplicates; discuss leakage in the report:\',duplicate_pairs)\n    fingerprint=hashlib.sha256(json.dumps({\'sources\':{s:sha256(p) for s,p in sources.items()},\n        \'image_hashes\':{s:sorted(h) for s,h in split_hashes.items()},\'classes\':names},sort_keys=True).encode()).hexdigest()\n    report=dict(dataset_url=DATASET_URL,counts=counts,classes=names,class_counts=class_counts,\n                exact_cross_split_duplicates=duplicate_pairs,dataset_fingerprint=fingerprint,\n                source_json_hashes={s:sha256(p) for s,p in sources.items()})\n    write_json(out/\'dataset_report.json\',report); write_json(out/\'classes.json\',names)\n    print(\'Classes:\',len(names),\'| original splits:\',counts)\n    return dict(root=str(out),names=names,paths=prepared,fingerprint=fingerprint)\n\ndef coco_to_yolo(prepared):\n    """Convert polygons only for YOLO training. Evaluate on untouched COCO masks."""\n    import yaml\n    from ultralytics.data.converter import merge_multi_segment\n    root=Path(prepared[\'root\'])/\'yolo\'; multi=0\n    for split,json_path in prepared[\'paths\'].items():\n        raw=read_json(json_path); by=defaultdict(list)\n        for a in raw[\'annotations\']:\n            if not a[\'iscrowd\']: by[a[\'image_id\']].append(a)\n        imgdir=root/\'images\'/split; labdir=root/\'labels\'/split\n        imgdir.mkdir(parents=True,exist_ok=True); labdir.mkdir(parents=True,exist_ok=True)\n        for rec in raw[\'images\']:\n            source=Path(rec[\'file_name\']); dest=imgdir/f\'{rec["id"]}{source.suffix.lower()}\'\n            if not dest.exists(): dest.symlink_to(source)\n            lines=[]\n            for a in by[rec[\'id\']]:\n                seg=a[\'segmentation\']\n                if not isinstance(seg,list):\n                    raise ValueError(\'YOLO conversion needs polygon export. RLE masks cannot be losslessly \'\n                                     \'represented in YOLO polygon labels. Re-export as COCO Segmentation polygons.\')\n                if len(seg)>1:\n                    points=np.concatenate(merge_multi_segment(seg),axis=0); multi+=1\n                else: points=np.asarray(seg[0],dtype=float).reshape(-1,2)\n                points=points / np.array([rec[\'width\'],rec[\'height\']])\n                if not np.isfinite(points).all(): raise ValueError(\'Non-finite polygon coordinates.\')\n                if (points < -1e-5).any() or (points > 1.00001).any():\n                    raise ValueError(\'Out-of-image polygon: repair source labels before training.\')\n                points=np.clip(points,0,1)\n                lines.append(str(a[\'category_id\']-1)+\' \'+\' \'.join(f\'{v:.7f}\' for v in points.ravel()))\n            (labdir/f\'{rec["id"]}.txt\').write_text(\'\\n\'.join(lines),encoding=\'utf-8\')\n    config={\'path\':str(root.resolve()),\'train\':\'images/train\',\'val\':\'images/valid\',\'test\':\'images/test\',\'names\':prepared[\'names\']}\n    (root/\'data.yaml\').write_text(yaml.safe_dump(config,sort_keys=False),encoding=\'utf-8\')\n    write_json(root/\'conversion_report.json\',{\'multipolygon_instances_bridged\':multi,\n        \'note\':\'YOLO uses connected polygons for multipart objects; COCO evaluation retains the original masks. Crowd labels excluded from training.\'})\n    return str(root/\'data.yaml\')\n\nclass IdentityAlbumentations:\n    def __init__(self,*args,**kwargs): self.transform=None; self.p=0.\n    def __call__(self,labels): return labels\n\n@contextlib.contextmanager\ndef yolo_augmentation_policy(regime):\n    import ultralytics.data.augment as aug\n    original=aug.Albumentations\n    # Native default includes four p=.01 photometric Albumentations operations.\n    # Disable the hidden photometric pipeline for none and custom.\n    if regime!=\'default\': aug.Albumentations=IdentityAlbumentations\n    else:\n        import albumentations  # fail instead of silently changing the default policy\n        if original().transform is None: raise RuntimeError(\'Default Albumentations failed to initialize.\')\n    try: yield\n    finally: aug.Albumentations=original\n\ndef collect_environment(run_dir):\n    import torch, torchvision\n    write_json(Path(run_dir)/\'environment.json\',dict(python=sys.version,torch=torch.__version__,\n        torchvision=torchvision.__version__,gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else \'CPU\',\n        cuda=torch.version.cuda,seed=SEED))\n    p=subprocess.run([sys.executable,\'-m\',\'pip\',\'freeze\'],capture_output=True,text=True)\n    (Path(run_dir)/\'pip_freeze.txt\').write_text(p.stdout,encoding=\'utf-8\')\n\ndef plot_predictions(rec, predictions, categories, out, threshold=.25):\n    im=Image.open(rec[\'file_name\']).convert(\'RGB\'); overlay=Image.new(\'RGBA\',im.size,(0,0,0,0)); draw=ImageDraw.Draw(overlay)\n    palette=[(255,90,70),(50,200,255),(170,90,255),(255,210,40),(40,225,130)]\n    for p in predictions:\n        if p[\'score\']<threshold: continue\n        m=decode_segmentation(p[\'segmentation\'],rec[\'height\'],rec[\'width\'])\n        color=palette[(p[\'category_id\']-1)%len(palette)]\n        rgba=Image.new(\'RGBA\',im.size,color+(95,))\n        overlay.alpha_composite(Image.composite(rgba,Image.new(\'RGBA\',im.size),Image.fromarray(m*255)))\n        x,y,w,h=p[\'bbox\']; draw=ImageDraw.Draw(overlay)\n        draw.rectangle([x,y,x+w,y+h],outline=color+(255,),width=2)\n        draw.text((x,max(0,y-12)),f\'{categories[p["category_id"]]} {p["score"]:.2f}\',fill=color+(255,))\n    Image.alpha_composite(im.convert(\'RGBA\'),overlay).convert(\'RGB\').save(out)\n\ndef evaluate_coco(json_path,predictions,out_dir,visual_limit=12):\n    from pycocotools.coco import COCO\n    from pycocotools.cocoeval import COCOeval\n    out=Path(out_dir); out.mkdir(parents=True,exist_ok=True)\n    gt=COCO(str(json_path)); write_json(out/\'predictions_coco.json\',predictions)\n    # Remove bbox for segmentation loadRes so size-area bins use mask area.\n    dt=gt.loadRes([{k:v for k,v in p.items() if k!=\'bbox\'} for p in predictions]) if predictions else COCO()\n    if not predictions:\n        dt.dataset=dict(images=gt.dataset[\'images\'],categories=gt.dataset[\'categories\'],annotations=[]);dt.createIndex()\n    ev=COCOeval(gt,dt,\'segm\'); ev.params.maxDets=[1,10,100]\n    log=io.StringIO()\n    with contextlib.redirect_stdout(log): ev.evaluate();ev.accumulate();ev.summarize()\n    print(log.getvalue()); (out/\'coco_summary.txt\').write_text(log.getvalue())\n    metrics={k:float(ev.stats[i]) for i,k in enumerate([\'mask_AP50_95\',\'mask_AP50\',\'mask_AP75\',\n        \'mask_AP_small\',\'mask_AP_medium\',\'mask_AP_large\',\'mask_AR1\',\'mask_AR10\',\'mask_AR100\',\n        \'mask_AR_small\',\'mask_AR_medium\',\'mask_AR_large\'])}\n    metrics.update(images=len(gt.imgs),instances=len(gt.anns),max_detections=100)\n    write_json(out/\'metrics.json\',metrics)\n    with open(out/\'per_class_AP.csv\',\'w\',newline=\'\') as f:\n        writer=csv.writer(f); writer.writerow([\'category_id\',\'class\',\'mask_AP50_95\',\'mask_AP50\',\'gt_instances\'])\n        for k,cid in enumerate(ev.params.catIds):\n            all_ap=ev.eval[\'precision\'][:,:,k,0,-1]; ap50=ev.eval[\'precision\'][0,:,k,0,-1]\n            writer.writerow([cid,gt.cats[cid][\'name\'],float(all_ap[all_ap>=0].mean()) if (all_ap>=0).any() else \'\',\n                             float(ap50[ap50>=0].mean()) if (ap50>=0).any() else \'\',len(gt.getAnnIds(catIds=[cid]))])\n    if visual_limit:\n        vis=out/\'overlays\';vis.mkdir(exist_ok=True); by=defaultdict(list)\n        for p in predictions: by[p[\'image_id\']].append(p)\n        for rec in list(gt.imgs.values())[:visual_limit]:\n            plot_predictions(rec,by[rec[\'id\']],{k:v[\'name\'] for k,v in gt.cats.items()},vis/f\'{rec["id"]}.jpg\')\n    return metrics\n\ndef predict_yolo(weights,json_path,imgsz=640):\n    from ultralytics import YOLO\n    from pycocotools import mask as mu\n    model=YOLO(str(weights)); predictions=[]\n    for rec in read_json(json_path)[\'images\']:\n        r=model.predict(rec[\'file_name\'],imgsz=imgsz,device=0,conf=.001,iou=.7,\n                        max_det=100,retina_masks=True,verbose=False)[0]\n        if r.masks is None: continue\n        for mask,score,cls in zip(r.masks.data.cpu().numpy(),r.boxes.conf.cpu().numpy(),r.boxes.cls.cpu().numpy()):\n            if mask.shape!=(rec[\'height\'],rec[\'width\']): raise RuntimeError(\'YOLO mask was not restored to source resolution.\')\n            m=(mask>.5).astype(np.uint8); enc=encode_mask(m)\n            predictions.append(dict(image_id=rec[\'id\'],category_id=int(cls)+1,score=float(score),\n                                    segmentation=enc,bbox=mu.toBbox(enc).tolist()))\n    return predictions\n\n# Torch imports are local except the Dataset base; notebook installs dependencies before importing.\nimport torch\nfrom torch.utils.data import Dataset,DataLoader\n\nclass CocoInstances(Dataset):\n    def __init__(self,json_path,regime=\'none\',training=False):\n        self.raw=read_json(json_path);self.images=self.raw[\'images\'];self.training=training;self.regime=regime\n        self.by=defaultdict(list)\n        for a in self.raw[\'annotations\']:\n            if not a[\'iscrowd\']: self.by[a[\'image_id\']].append(a)\n    def __len__(self): return len(self.images)\n    def __getitem__(self,i):\n        from torchvision.transforms import functional as F, InterpolationMode\n        rec=self.images[i]; im=Image.open(rec[\'file_name\']).convert(\'RGB\'); w,h=im.size\n        anns=self.by[rec[\'id\']]\n        masks=np.stack([decode_segmentation(a[\'segmentation\'],h,w) for a in anns]) if anns else np.zeros((0,h,w),np.uint8)\n        masks=torch.from_numpy(masks); labels=torch.tensor([a[\'category_id\'] for a in anns],dtype=torch.int64)\n        if self.training:\n            if self.regime in (\'default\',\'custom\') and random.random()<.5:\n                im=F.hflip(im);masks=masks.flip(-1)\n            if self.regime==\'custom\':\n                # Geometry uses exactly the same transform and nearest-neighbor masks.\n                angle=random.uniform(-10,10)\n                im=F.rotate(im,angle,InterpolationMode.BILINEAR,fill=0)\n                if len(masks): masks=F.rotate(masks,angle,InterpolationMode.NEAREST,fill=0)\n                im=F.adjust_brightness(im,random.uniform(.5,1.5))\n                im=F.adjust_contrast(im,random.uniform(.7,1.3))\n                im=F.adjust_saturation(im,random.uniform(.7,1.3))\n                if random.random()<.3: im=F.adjust_gamma(im,random.uniform(.7,1.6))\n        # Recompute boxes from transformed masks (max coordinate +1, including one-pixel objects).\n        boxes=[]; kept=[]\n        for j,m in enumerate(masks):\n            ys,xs=torch.where(m>0)\n            if len(xs): boxes.append([xs.min(),ys.min(),xs.max()+1,ys.max()+1]); kept.append(j)\n        masks=masks[kept];labels=labels[kept]\n        boxes=torch.tensor(boxes,dtype=torch.float32).reshape(-1,4)\n        target=dict(boxes=boxes,labels=labels,masks=masks,image_id=torch.tensor(rec[\'id\']),\n                    area=masks.sum((1,2)).float(),iscrowd=torch.zeros(len(kept),dtype=torch.int64))\n        return F.to_tensor(im),target\n\ndef collate(batch): return tuple(zip(*batch))\n\ndef preview_maskrcnn_dataset(dataset, classes, output_dir, limit=6):\n    """Save transformed image/mask pairs as real augmentation-process evidence."""\n    from torchvision.transforms.functional import to_pil_image\n    from pycocotools import mask as mu\n    out=Path(output_dir);out.mkdir(parents=True,exist_ok=True)\n    for i in range(min(limit,len(dataset))):\n        image,target=dataset[i];path=out/f\'{i:02d}_image.jpg\';to_pil_image(image).save(path)\n        rec=dict(file_name=str(path),height=image.shape[1],width=image.shape[2])\n        preds=[]\n        for mask,label in zip(target[\'masks\'],target[\'labels\']):\n            rle=encode_mask(mask.numpy())\n            preds.append(dict(category_id=int(label),score=1.,segmentation=rle,bbox=mu.toBbox(rle).tolist()))\n        plot_predictions(rec,preds,{i+1:n for i,n in enumerate(classes)},out/f\'{i:02d}_ground_truth_masks.jpg\')\n\ndef plot_maskrcnn_history(run):\n    import matplotlib.pyplot as plt\n    with open(Path(run)/\'results.csv\',newline=\'\') as f: rows=list(csv.DictReader(f))\n    if not rows:return\n    x=[int(r[\'epoch\']) for r in rows]\n    fig,axes=plt.subplots(1,2,figsize=(10,4))\n    axes[0].plot(x,[float(r[\'train_loss\']) for r in rows]);axes[0].set_title(\'Training loss\')\n    axes[1].plot(x,[float(r[\'val_mask_AP50_95\']) for r in rows],label=\'mask AP50–95\')\n    axes[1].plot(x,[float(r[\'val_mask_AP50\']) for r in rows],label=\'mask AP50\');axes[1].legend()\n    axes[1].set_title(\'Original validation\');axes[1].set_ylim(0,1)\n    for ax in axes:ax.set_xlabel(\'Epoch\');ax.grid(alpha=.2)\n    fig.tight_layout();fig.savefig(Path(run)/\'training_curves.png\',dpi=150);plt.close(fig)\n\ndef build_maskrcnn(nclasses,imgsz=640,pretrained=True):\n    from torchvision.models.detection import maskrcnn_resnet50_fpn,MaskRCNN_ResNet50_FPN_Weights\n    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor\n    from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor\n    # Aspect ratio preserved: shorter side targets imgsz, longer side capped at imgsz.\n    model=maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT if pretrained else None,\n        weights_backbone=None,min_size=imgsz,max_size=imgsz,box_score_thresh=.001,box_detections_per_img=100)\n    model.roi_heads.box_predictor=FastRCNNPredictor(model.roi_heads.box_predictor.cls_score.in_features,nclasses+1)\n    model.roi_heads.mask_predictor=MaskRCNNPredictor(model.roi_heads.mask_predictor.conv5_mask.in_channels,256,nclasses+1)\n    return model\n\n@torch.inference_mode()\ndef predict_maskrcnn(model,json_path,device=\'cuda\'):\n    from pycocotools import mask as mu\n    model.eval();predictions=[]\n    ds=CocoInstances(json_path)\n    for image,target in ds:\n        pred=model([image.to(device)])[0]\n        for m,score,label in zip(pred[\'masks\'][:,0].cpu().numpy(),pred[\'scores\'].cpu().numpy(),pred[\'labels\'].cpu().numpy()):\n            if score<.001: continue\n            enc=encode_mask(m>=.5)\n            predictions.append(dict(image_id=int(target[\'image_id\']),category_id=int(label),score=float(score),\n                                    segmentation=enc,bbox=mu.toBbox(enc).tolist()))\n    return predictions\n\ndef save_checkpoint(path,state):\n    tmp=Path(str(path)+\'.tmp\');torch.save(state,tmp);tmp.replace(path)\n\ndef train_maskrcnn(prepared,run,regime,epochs=30,imgsz=640,batch=2,workers=2,lr=.0025):\n    seed_all();device=\'cuda\';model=build_maskrcnn(len(prepared[\'names\']),imgsz).to(device)\n    optimizer=torch.optim.SGD([p for p in model.parameters() if p.requires_grad],lr=lr,momentum=.9,weight_decay=.0005)\n    scheduler=torch.optim.lr_scheduler.MultiStepLR(optimizer,milestones=sorted(set([int(epochs*.67),int(epochs*.89)])),gamma=.1)\n    scaler=torch.amp.GradScaler(\'cuda\'); generator=torch.Generator().manual_seed(SEED)\n    ds=CocoInstances(prepared[\'paths\'][\'train\'],regime,True)\n    if not (run/\'augmentation_previews\').exists():\n        preview_maskrcnn_dataset(ds,prepared[\'names\'],run/\'augmentation_previews\')\n    seed_all()  # Preview sampling must not shift the training RNG stream, even after a retry.\n    loader=DataLoader(ds,batch_size=batch,shuffle=True,num_workers=workers,collate_fn=collate,\n                      pin_memory=True,worker_init_fn=seed_worker,generator=generator,persistent_workers=False)\n    start=0; best=-1.; elapsed=0.; last=run/\'last.pth\'\n    if last.exists():\n        ck=torch.load(last,map_location=\'cpu\',weights_only=False)  # only your own trusted checkpoint\n        model.load_state_dict(ck[\'model\']);optimizer.load_state_dict(ck[\'optimizer\']);scheduler.load_state_dict(ck[\'scheduler\'])\n        scaler.load_state_dict(ck[\'scaler\']);start=ck[\'epoch\'];best=ck[\'best_AP\'];elapsed=ck[\'train_val_seconds\']\n        random.setstate(ck[\'rng_python\']);np.random.set_state(ck[\'rng_numpy\']);torch.set_rng_state(ck[\'rng_torch\'])\n        torch.cuda.set_rng_state_all(ck[\'rng_cuda\']);generator.set_state(ck[\'rng_loader\'])\n    logpath=run/\'results.csv\'\n    # Remove any log row written after the latest successfully committed checkpoint.\n    if logpath.exists():\n        with open(logpath,newline=\'\') as f: rows=list(csv.DictReader(f));fields=list(rows[0]) if rows else []\n        if fields:\n            with open(logpath,\'w\',newline=\'\') as f:\n                writer=csv.DictWriter(f,fields);writer.writeheader();writer.writerows(r for r in rows if int(r[\'epoch\'])<=start)\n    for epoch in range(start,epochs):\n        torch.cuda.synchronize();t=time.perf_counter();model.train();totals=defaultdict(float)\n        for images,targets in loader:\n            images=[x.to(device,non_blocking=True) for x in images]\n            targets=[{k:v.to(device) for k,v in target.items()} for target in targets]\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=\'cuda\',dtype=torch.float16):\n                losses=model(images,targets);loss=sum(losses.values())\n            if not torch.isfinite(loss): raise FloatingPointError(f\'Nonfinite training loss at epoch {epoch+1}\')\n            scaler.scale(loss).backward();scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(model.parameters(),10.);scaler.step(optimizer);scaler.update()\n            for k,v in losses.items(): totals[k]+=float(v.detach())\n            totals[\'train_loss\']+=float(loss.detach())\n        learning_rate=optimizer.param_groups[0][\'lr\'];scheduler.step()\n        metrics=evaluate_coco(prepared[\'paths\'][\'valid\'],predict_maskrcnn(model,prepared[\'paths\'][\'valid\']),run/\'validation_latest\',0)\n        torch.cuda.synchronize();seconds=time.perf_counter()-t;elapsed+=seconds\n        row=dict(epoch=epoch+1,**{k:v/len(loader) for k,v in totals.items()},\n                 val_mask_AP50_95=metrics[\'mask_AP50_95\'],val_mask_AP50=metrics[\'mask_AP50\'],lr=learning_rate,\n                 epoch_seconds=seconds,train_val_seconds=elapsed)\n        with open(logpath,\'a\',newline=\'\') as f:\n            writer=csv.DictWriter(f,fieldnames=list(row))\n            if f.tell()==0: writer.writeheader()\n            writer.writerow(row)\n        score=metrics[\'mask_AP50_95\']\n        if score>best:\n            best=score\n            save_checkpoint(run/\'best.pth\',dict(model=model.state_dict(),classes=prepared[\'names\'],\n                                               epoch=epoch+1,imgsz=imgsz,best_AP=best))\n        save_checkpoint(last,dict(model=model.state_dict(),optimizer=optimizer.state_dict(),scheduler=scheduler.state_dict(),\n            scaler=scaler.state_dict(),epoch=epoch+1,best_AP=best,train_val_seconds=elapsed,classes=prepared[\'names\'],imgsz=imgsz,\n            rng_python=random.getstate(),rng_numpy=np.random.get_state(),rng_torch=torch.get_rng_state(),\n            rng_cuda=torch.cuda.get_rng_state_all(),rng_loader=generator.get_state()))\n        print(f\'Epoch {epoch+1}/{epochs}: loss={row["train_loss"]:.4f}, val mask AP={score:.4f}, {seconds:.1f}s\')\n    del model;torch.cuda.empty_cache()\n    plot_maskrcnn_history(run)\n    return run/\'best.pth\',elapsed\n\ndef completed_yolo_fit_time(run, epochs):\n    """Recover a completed fit if interruption occurred before wrapper bookkeeping."""\n    run=Path(run);last=run/\'fit/weights/last.pt\';best=run/\'fit/weights/best.pt\';csvpath=run/\'fit/results.csv\'\n    if not all(p.is_file() for p in (last,best,csvpath)): return None\n    with open(csvpath,newline=\'\') as f:\n        rows=[{k.strip():v for k,v in r.items()} for r in csv.DictReader(f)]\n    if not rows or int(float(rows[-1][\'epoch\']))<epochs: return None\n    ck=torch.load(last,map_location=\'cpu\',weights_only=False)  # own trusted checkpoint only\n    epoch=ck.get(\'epoch\',-2)\n    committed=epoch>=epochs-1 or (epoch==-1 and ck.get(\'optimizer\') is None)\n    return float(rows[-1][\'time\']) if committed else None\n\ndef train_yolo(prepared,run,model_name,regime,epochs=30,imgsz=640,batch=4,workers=2,lr=.001):\n    import ultralytics\n    from ultralytics import YOLO\n    if ultralytics.__version__!=\'8.3.221\': raise RuntimeError(\'Install the pinned ultralytics==8.3.221 version.\')\n    data=coco_to_yolo(prepared);seed_all();last=run/\'fit/weights/last.pt\'\n    # A completed fit can be evaluated again without asking YOLO to resume a stripped checkpoint.\n    fitdone=run/\'fit_complete.json\'\n    if fitdone.exists(): return run/\'fit/weights/best.pt\',read_json(fitdone)[\'train_val_seconds\']\n    recovered_time=completed_yolo_fit_time(run,epochs)\n    if recovered_time is not None:\n        write_json(fitdone,dict(train_val_seconds=recovered_time,recovered_completed_fit=True))\n        return run/\'fit/weights/best.pt\',recovered_time\n    kwargs=dict(data=data,epochs=epochs,imgsz=imgsz,batch=batch,device=0,workers=workers,\n                optimizer=\'AdamW\',lr0=lr,lrf=.01,weight_decay=.0005,seed=SEED,deterministic=True,\n                patience=0,amp=True,cache=False,save=True,save_period=5,plots=True,\n                project=str(run),name=\'fit\',exist_ok=True,pretrained=True,\n                multi_scale=False,rect=False,**YOLO_AUG[regime])\n    # Keep augmentation policy active during dataset construction, including a resumed run.\n    with yolo_augmentation_policy(regime):\n        model=YOLO(str(last) if last.exists() else model_name+\'.pt\')\n        torch.cuda.synchronize();t=time.perf_counter()\n        if last.exists(): model.train(resume=True,device=0,workers=workers)\n        else: model.train(**kwargs)\n        torch.cuda.synchronize();session_seconds=time.perf_counter()-t\n    # Native CSV time is cumulative within YOLO\'s recorded fit (including epoch validation).\n    with open(run/\'fit/results.csv\',newline=\'\') as f: rows=[{k.strip():v for k,v in r.items()} for r in csv.DictReader(f)]\n    native_seconds=float(rows[-1].get(\'time\',session_seconds))\n    write_json(fitdone,dict(train_val_seconds=native_seconds,current_session_seconds=session_seconds,\n                          note=\'Native results.csv time; interrupted YOLO sessions may reset its time column. See session ledger.\'))\n    del model;torch.cuda.empty_cache()\n    return run/\'fit/weights/best.pt\',native_seconds\n\ndef run_experiment(prepared,out_root,model_name,regime,epochs=30,imgsz=640,\n                   yolo_batch=4,mask_batch=2,workers=2):\n    require_gpu()\n    if model_name not in MODELS or regime not in REGIMES: raise ValueError(\'Unknown model/augmentation policy.\')\n    if not 1<=epochs<=1000: raise ValueError(\'Invalid epoch count.\')\n    run=Path(out_root)/f\'{model_name}_{regime}\';run.mkdir(parents=True,exist_ok=True)\n    config=dict(model=model_name,regime=regime,epochs=epochs,imgsz=imgsz,seed=SEED,\n                batch=mask_batch if model_name==\'maskrcnn\' else yolo_batch,\n                dataset_fingerprint=prepared[\'fingerprint\'],classes=prepared[\'names\'],\n                augmentation=YOLO_AUG[regime] if model_name!=\'maskrcnn\' else regime,\n                selection=\'Original validation set only; no test-based checkpoint selection\')\n    if (run/\'config.json\').exists() and read_json(run/\'config.json\')!=config:\n        raise ValueError(f\'Run settings changed: {run}. Use a new OUT_ROOT instead of mixing incompatible experiments.\')\n    write_json(run/\'config.json\',config)\n    for filename in (\'dataset_report.json\',\'classes.json\'):\n        shutil.copy2(Path(prepared[\'root\'])/filename,run/filename)\n    if (run/\'completed.json\').exists():\n        print(\'Already complete:\',run.name);return read_json(run/\'completed.json\')\n    collect_environment(run);session_start=time.time();status=\'interrupted_or_failed\'\n    try:\n        if model_name==\'maskrcnn\':\n            weights,seconds=train_maskrcnn(prepared,run,regime,epochs,imgsz,mask_batch,workers)\n            ck=torch.load(weights,map_location=\'cpu\',weights_only=False)\n            model=build_maskrcnn(len(prepared[\'names\']),imgsz,False).cuda();model.load_state_dict(ck[\'model\'])\n            t=time.perf_counter();pred=predict_maskrcnn(model,prepared[\'paths\'][\'test\'])\n            del model\n        else:\n            weights,seconds=train_yolo(prepared,run,model_name,regime,epochs,imgsz,yolo_batch,workers)\n            t=time.perf_counter();pred=predict_yolo(weights,prepared[\'paths\'][\'test\'],imgsz)\n        metrics=evaluate_coco(prepared[\'paths\'][\'test\'],pred,run/\'original_test\')\n        result=dict(model=model_name,regime=regime,epochs=epochs,weights=str(weights),\n                    train_val_seconds=seconds,test_seconds=time.perf_counter()-t,**metrics)\n        write_json(run/\'completed.json\',result);status=\'complete\'\n        return result\n    finally:\n        torch.cuda.empty_cache()\n        with open(run/\'session_ledger.jsonl\',\'a\') as f:\n            f.write(json.dumps(dict(start_unix=session_start,end_unix=time.time(),\n                allocated_session_seconds=time.time()-session_start,gpu_count=1,status=status))+\'\\n\')\n        # Ledger is wall time while a GPU is allocated, not measured kernel-only GPU time.\n\ndef prepare_own_test(json_path,run_dir,out_dir):\n    names=read_json(Path(run_dir)/\'classes.json\');canonical,hashes=canonicalize(json_path,names)\n    if len(canonical[\'images\'])<40: print(\'WARNING: assignment requires at least 40 own photos; this export has\',len(canonical[\'images\']))\n    out=Path(out_dir);out.mkdir(parents=True,exist_ok=True);p=out/\'own_test_canonical.json\';write_json(p,canonical)\n    write_json(out/\'own_test_manifest.json\',dict(source=str(json_path),source_sha256=sha256(json_path),image_hashes=hashes,\n                                               classes=names,note=\'Never used in training or checkpoint selection.\'))\n    return p\n\ndef evaluate_own(run_dir,own_json,out_dir=None,imgsz_override=None):\n    require_gpu();run=Path(run_dir);cfg=read_json(run/\'config.json\')\n    out=Path(out_dir) if out_dir else Path(\'/kaggle/working/own_evaluation\')/run.name\n    json_path=prepare_own_test(own_json,run,out);size=imgsz_override or cfg[\'imgsz\']\n    if cfg[\'model\']==\'maskrcnn\':\n        ck=torch.load(run/\'best.pth\',map_location=\'cpu\',weights_only=False)\n        if ck[\'classes\']!=cfg[\'classes\']: raise ValueError(\'Checkpoint class mapping mismatch.\')\n        model=build_maskrcnn(len(cfg[\'classes\']),size,False).cuda();model.load_state_dict(ck[\'model\'])\n        pred=predict_maskrcnn(model,json_path);del model\n    else: pred=predict_yolo(run/\'fit/weights/best.pt\',json_path,size)\n    metrics=evaluate_coco(json_path,pred,out,visual_limit=100000)\n    if (run/\'original_test/metrics.json\').exists() and not imgsz_override:\n        original=read_json(run/\'original_test/metrics.json\')\n        write_json(out/\'domain_shift.json\',dict(original_AP=original[\'mask_AP50_95\'],own_AP=metrics[\'mask_AP50_95\'],\n            drop_percentage_points=100*(original[\'mask_AP50_95\']-metrics[\'mask_AP50_95\'])))\n    write_json(out/\'evaluation_config.json\',dict(run=str(run),imgsz=size,diagnostic_override=bool(imgsz_override),\n        warning=\'A changed input size on own test is a post-hoc diagnostic, not the primary held-out score.\' if imgsz_override else \'\'))\n    torch.cuda.empty_cache();return metrics\n\ndef comparison_csv(out_root):\n    rows=[]\n    for p in sorted(Path(out_root).glob(\'*/completed.json\')):\n        d=read_json(p);rows.append({k:d[k] for k in (\'model\',\'regime\',\'epochs\',\'mask_AP50_95\',\'mask_AP50\',\'mask_AP75\',\'train_val_seconds\',\'test_seconds\')})\n    if not rows: return None\n    dest=Path(out_root)/\'comparison_original_test.csv\'\n    with open(dest,\'w\',newline=\'\') as f:\n        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)\n    return dest\n\ndef archive_run(run_dir,destination):\n    run=Path(run_dir);destination=Path(destination);destination.parent.mkdir(parents=True,exist_ok=True)\n    return shutil.make_archive(str(destination.with_suffix(\'\')),\'zip\',root_dir=run.parent,base_dir=run.name)\n', encoding="utf-8")
(CODE_DIR / "reference_classes.txt").write_text('backpack\t\n\t\n\tbed\t\n\t\n\tblanket\t\n\t\n\tbook\t\n\t\n\tbottle\t\n\t\n\tbowl\t\n\t\n\tbox\t\n\t\n\tcabinet-merged\t\n\t\n\tcardboard\t\n\t\n\tceiling-merged\t\n\t\n\tcell phone\t\n\t\n\tchair\t\n\t\n\tclock\t\n\t\n\tcouch\t\n\t\n\tcounter\t\n\t\n\tcup\t\n\t\n\tcurtain\t\n\t\n\tdining table\t\n\t\n\tdoor-stuff\t\n\t\n\tfire-extinguisher\t\n\t\n\tfloor-other-merged\t\n\t\n\tfloor-wood\t\n\t\n\tflower\t\n\t\n\tfood-other-merged\t\n\t\n\tfork\t\n\t\n\thair drier\t\n\t\n\thandbag\t\n\t\n\tkeyboard\t\n\t\n\tknife\t\n\t\n\tlaptop\t\n\t\n\tlight\t\n\t\n\tmicrowave\t\n\t\n\tmirror-stuff\t\n\t\n\tmouse\t\n\t\n\toven\t\n\t\n\tpan\t\n\t\n\tpaper\t\n\t\n\tpaper-merged\t\n\t\n\tperson\t\n\t\n\tpillow\t\n\t\n\tplate\t\n\t\n\tplug\t\n\t\n\tpotted plant\t\n\t\n\trefrigerator\t\n\t\n\tremote\t\n\t\n\trug-merged\t\n\t\n\tscissor\t\n\t\n\tscissors\t\n\t\n\tshelf\t\n\t\n\tsink\t\n\t\n\tspoon\t\n\t\n\tst\\\t\n\t\n\tstairs\t\n\t\n\tstove-merged\t\n\t\n\ttable-merged\t\n\t\n\tteddy bear\t\n\t\n\ttoaster\t\n\t\n\ttoilet\t\n\t\n\ttoothbrush\t\n\t\n\ttowel\t\n\t\n\ttoy\t\n\t\n\ttrash-bin\t\n\t\n\ttree-merged\t\n\t\n\ttv\t\n\t\n\tvase\t\n\t\n\twall\t\n\t\n\twall-brick\t\n\t\n\twall-other-merged\t\n\t\n\twall-stone\t\n\t\n\twall-tile\t\n\t\n\twall-wood\t\n\t\n\twallet\t\n\t\n\twindow-blind\t\n\t\n\twindow-other', encoding="utf-8")
if str(CODE_DIR) not in sys.path: sys.path.insert(0, str(CODE_DIR))
import seg046
s = importlib.reload(seg046)
print("Training/evaluation helpers loaded.")


## 4. Preflight and dataset validation

In [ ]:
s.require_gpu()
source_root = s.obtain_dataset(DATA_ROOT)
prepared = s.prepare_data(source_root, PREPARED_ROOT, strict_v5=STRICT_V5)
# Report tag-list discrepancies without silently changing the actual export's labels.
reference = {line.strip() for line in (CODE_DIR / "reference_classes.txt").read_text().splitlines() if line.strip()}
actual = set(prepared["names"])
print("Reference tags absent from export:", sorted(reference - actual))
print("Export classes absent from reference tags:", sorted(actual - reference))
print("Resolved exact class order:", prepared["names"])
from IPython.display import display
import pandas as pd
report = s.read_json(Path(PREPARED_ROOT) / "dataset_report.json")
display(pd.DataFrame(report["class_counts"]).fillna(0).astype(int))


## 5. Optional restore from a previous session

In [ ]:
import shutil
Path(OUT_ROOT).mkdir(parents=True, exist_ok=True)
if RESTORE_RUNS_FROM:
    old_root = Path(RESTORE_RUNS_FROM)
    allowed = {f"{model}_{regime}" for model in s.MODELS for regime in s.REGIMES}
    candidates = [p.parent for p in old_root.rglob("config.json") if p.parent.name in allowed]
    if not candidates: raise FileNotFoundError("No saved run folders found under RESTORE_RUNS_FROM.")
    for folder in candidates:
        dest = Path(OUT_ROOT) / folder.name
        if dest.exists():
            print("Keeping existing writable run:", dest)
        else:
            shutil.copytree(folder, dest)
            print("Restored:", dest)


## 6. Train, evaluate original test, save each result

In [ ]:
from IPython.display import display, FileLink
import time
zip_dir = Path("/kaggle/working/result_zips")
zip_dir.mkdir(exist_ok=True)
results = []
for regime in REGIMES:
    print(f"\n===== {MODEL} | {regime} | {EPOCHS} epochs =====")
    run_dir = Path(OUT_ROOT) / f"{MODEL}_{regime}"
    try:
        result = s.run_experiment(prepared, OUT_ROOT, MODEL, regime,
            epochs=EPOCHS, imgsz=IMGSZ, yolo_batch=YOLO_BATCH,
            mask_batch=MASK_BATCH, workers=WORKERS)
        results.append(result)
    finally:
        # Make a partial checkpoint archive even if this Python run raises an error.
        # A forced process/session kill can bypass this block; retain Kaggle outputs.
        if run_dir.exists():
            archive = s.archive_run(run_dir, zip_dir / f"{run_dir.name}.zip")
            print("Saved", archive)
            display(FileLink(archive))
    table = s.comparison_csv(OUT_ROOT)
    if table:
        display(pd.read_csv(table))
print("Requested runs finished. Save this notebook version and download the result ZIPs.")


## Keep these outputs

Download every ZIP in `/kaggle/working/result_zips`, or retain the full saved notebook outputs. Each contains checkpoints, training logs, configuration, class mapping, provenance and original-test scores. `completed.json` means the complete fit + original-test evaluation finished; a partial ZIP is not a completed experiment.

Your own photos are intentionally absent. When their polygons are ready, use **04_Evaluate_Own_Photos_Later.ipynb** with these saved run folders. See README for the ≥40 photos, ≥10 difficult, annotation screenshots, EXIF, five failure cases and AI declaration requirements.

The scores above are measured output only after this notebook actually runs. The delivered notebook contains no invented accuracy.
